 Mask R-CNN — ResNet-50 FPN



## 1. Install dependencies

In [ ]:
import os, sys, subprocess
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
os.environ["WANDB_MODE"] = "disabled"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "ultralytics==8.3.221", "albumentations==2.0.8", "pycocotools==2.0.10",
    "roboflow==1.2.11", "PyYAML==6.0.2"])
# Keep Kaggle's matched CUDA torch/torchvision versions; do not install CPU wheels.


## 2. Configuration

In [ ]:
# For your planned experiments, change only EPOCHS.
# Dataset access still requires the Kaggle secret or an attached export.
MODEL = 'maskrcnn'
EPOCHS = 30                    # 25 is also fine: use the same value for all compared runs.
REGIMES = ["default", "none", "custom"]  # Run all three automatically: normal, none, custom.
IMGSZ = 640
YOLO_BATCH = 4
MASK_BATCH = 2
WORKERS = 2                   # Use 0 if Kaggle reports a multiprocessing error.
DATA_ROOT = ""                # Empty = automatic exact-v5 download using Kaggle secret.
STRICT_V5 = True
OUT_ROOT = "/kaggle/working/seg046/runs"
PREPARED_ROOT = "/kaggle/working/seg046/prepared"
RESTORE_RUNS_FROM = ""         # Optional /kaggle/input/... folder containing prior run folders.
# Restore only your own trusted checkpoints. For an uploaded ZIP, extract it into a
# working directory first, then point RESTORE_RUNS_FROM at its parent run directory.

## 3. Augmentation code — all three experiments
These are executable Python notebook cells, used directly by training below. No embedded `.py` script is written or imported.

All three experiments run automatically: **normal (`default`)**, **none**, and **custom**. You only need to change `EPOCHS` in Section 2 for your planned runs.

Custom uses only rotation ±10°, translation ±2% of width/height, scale 0.98–1.02, hue ±0.01 of a full hue cycle (about ±3.6°), and saturation/value multipliers 0.98–1.02. These retain the existing values. Flips, mosaic, mixup, copy-paste, CutMix, shear and perspective are disabled for custom.

Geometry transforms both the image and every instance mask; HSV changes the image only. Validation/test images receive no random augmentation. Required resizing/normalization still happens in each model.

In [ ]:
from __future__ import annotations
import contextlib, csv, hashlib, io, json, os, random, shutil, subprocess, sys, time
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F, InterpolationMode
SEED = 46
MODELS = ('yolo11n-seg', 'yolo11s-seg', 'maskrcnn')
VALID_REGIMES = ('none', 'default', 'custom')
DATASET_URL = 'https://universe.roboflow.com/panopticindoor/panoptic-indoor-segmentation/dataset/5'
EXPECTED_COUNTS = {'train': 397, 'valid': 166, 'test': 359}
YOLO_AUG = {}

### 3.1 None — no random augmentation

In [ ]:
YOLO_AUG['none'] = {'hsv_h': 0.0,
 'hsv_s': 0.0,
 'hsv_v': 0.0,
 'degrees': 0.0,
 'translate': 0.0,
 'scale': 0.0,
 'shear': 0.0,
 'perspective': 0.0,
 'flipud': 0.0,
 'fliplr': 0.0,
 'mosaic': 0.0,
 'mixup': 0.0,
 'copy_paste': 0.0,
 'cutmix': 0.0,
 'bgr': 0.0,
 'close_mosaic': 0}

def maskrcnn_none(image, masks):
    return image, masks

### 3.2 Normal (`default`) — existing baseline
YOLO keeps its existing native augmentation values below and its installed Albumentations pipeline (Blur, MedianBlur, ToGray, CLAHE, each with probability 0.01). Mask R-CNN keeps its existing 50% horizontal-flip baseline. These model-specific baselines are different; custom uses the same numeric limits for both.

In [ ]:
YOLO_AUG['default'] = {'hsv_h': 0.015,
 'hsv_s': 0.7,
 'hsv_v': 0.4,
 'degrees': 0.0,
 'translate': 0.1,
 'scale': 0.5,
 'shear': 0.0,
 'perspective': 0.0,
 'flipud': 0.0,
 'fliplr': 0.5,
 'mosaic': 1.0,
 'mixup': 0.0,
 'copy_paste': 0.0,
 'cutmix': 0.0,
 'bgr': 0.0,
 'close_mosaic': 10}

def maskrcnn_normal(image, masks):
    if random.random() < 0.5:
        image = F.hflip(image)
        masks = masks.flip(-1)
    return image, masks

### 3.3 Custom — rotation, translation, scale, HSV only

In [ ]:
# Fixed small changes requested for this experiment; no editing needed on Kaggle.
CUSTOM_AUG = dict(
    degrees=10.0,     # rotation from -10 to +10 degrees
    translate=0.02,   # up to 2% of width and height
    scale=0.02,       # scale factor from 0.98 to 1.02
    hsv_h=0.01,       # +/-1% of the hue cycle (about +/-3.6 degrees)
    hsv_s=0.02,       # saturation multiplier from 0.98 to 1.02
    hsv_v=0.02,       # value/brightness multiplier from 0.98 to 1.02
)
YOLO_AUG['custom'] = dict(
    **CUSTOM_AUG,
    shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.0,
    mosaic=0.0, mixup=0.0, copy_paste=0.0, cutmix=0.0,
    bgr=0.0, close_mosaic=0,
)

def custom_hsv(image,cfg):
    """Match Ultralytics 8.3.221 HSV ranges; operate on RGB image only."""
    import cv2
    gains=np.random.uniform(-1,1,3)*[cfg['hsv_h'],cfg['hsv_s'],cfg['hsv_v']]
    hue,sat,val=cv2.split(cv2.cvtColor(np.asarray(image),cv2.COLOR_RGB2HSV))
    x=np.arange(256,dtype=np.float32)
    lut_h=((x+gains[0]*180)%180).astype(np.uint8)
    lut_s=np.clip(x*(1+gains[1]),0,255).astype(np.uint8);lut_s[0]=0
    lut_v=np.clip(x*(1+gains[2]),0,255).astype(np.uint8)
    hsv=cv2.merge((cv2.LUT(hue,lut_h),cv2.LUT(sat,lut_s),cv2.LUT(val,lut_v)))
    return Image.fromarray(cv2.cvtColor(hsv,cv2.COLOR_HSV2RGB))


def maskrcnn_custom(image, masks):
    cfg = CUSTOM_AUG
    width, height = image.size
    angle = random.uniform(-cfg['degrees'], cfg['degrees'])
    shift = [round(random.uniform(-cfg['translate'], cfg['translate']) * width),
             round(random.uniform(-cfg['translate'], cfg['translate']) * height)]
    zoom = random.uniform(1 - cfg['scale'], 1 + cfg['scale'])
    # Reuse the SAME sampled geometry for image and instance masks.
    image = F.affine(image, angle, shift, zoom, [0., 0.],
                     InterpolationMode.BILINEAR, fill=114)
    if len(masks):
        masks = F.affine(masks, angle, shift, zoom, [0., 0.],
                         InterpolationMode.NEAREST, fill=0)
    image = custom_hsv(image, cfg)  # Color changes never touch masks.
    return image, masks

### 3.4 Connect the visible policies to training
Mask R-CNN dispatches to the three functions above and recomputes bounding boxes from the transformed masks. YOLO receives `YOLO_AUG[regime]` in its training call; its extra Albumentations pipeline is disabled for none/custom. Required model preprocessing is separate from these random augmentations.

In [ ]:
class IdentityAlbumentations:
    def __init__(self,*args,**kwargs): self.transform=None; self.p=0.
    def __call__(self,labels): return labels


@contextlib.contextmanager
def yolo_augmentation_policy(regime):
    import ultralytics.data.augment as aug
    original=aug.Albumentations
    # Native default includes four p=.01 photometric Albumentations operations.
    # Disable the hidden photometric pipeline for none and custom.
    if regime!='default': aug.Albumentations=IdentityAlbumentations
    else:
        import albumentations  # fail instead of silently changing the default policy
        if original().transform is None: raise RuntimeError('Default Albumentations failed to initialize.')
    try: yield
    finally: aug.Albumentations=original

In [ ]:
class CocoInstances(Dataset):
    def __init__(self,json_path,regime='none',training=False):
        self.raw=read_json(json_path);self.images=self.raw['images'];self.training=training;self.regime=regime
        self.by=defaultdict(list)
        for a in self.raw['annotations']:
            if not a['iscrowd']: self.by[a['image_id']].append(a)
    def __len__(self): return len(self.images)
    def __getitem__(self,i):
        from torchvision.transforms import functional as F, InterpolationMode
        rec=self.images[i]; im=Image.open(rec['file_name']).convert('RGB'); w,h=im.size
        anns=self.by[rec['id']]
        masks=np.stack([decode_segmentation(a['segmentation'],h,w) for a in anns]) if anns else np.zeros((0,h,w),np.uint8)
        masks=torch.from_numpy(masks); labels=torch.tensor([a['category_id'] for a in anns],dtype=torch.int64)
        if self.training:
            transform = {
                'none': maskrcnn_none,
                'default': maskrcnn_normal,
                'custom': maskrcnn_custom,
            }[self.regime]
            im, masks = transform(im, masks)
        # Recompute boxes from transformed masks (max coordinate +1, including one-pixel objects).
        boxes=[]; kept=[]
        for j,m in enumerate(masks):
            ys,xs=torch.where(m>0)
            if len(xs): boxes.append([xs.min(),ys.min(),xs.max()+1,ys.max()+1]); kept.append(j)
        masks=masks[kept];labels=labels[kept]
        boxes=torch.tensor(boxes,dtype=torch.float32).reshape(-1,4)
        target=dict(boxes=boxes,labels=labels,masks=masks,image_id=torch.tensor(rec['id']),
                    area=masks.sum((1,2)).float(),iscrowd=torch.zeros(len(kept),dtype=torch.int64))
        return F.to_tensor(im),target

### 3.5 Dataset preparation and utilities

In [ ]:
def write_json(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')
    tmp.replace(path)


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1024 * 1024), b''): h.update(b)
    return h.hexdigest()


def seed_all(seed=SEED):
    import torch
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    import torch
    s = torch.initial_seed() % 2**32
    random.seed(s); np.random.seed(s)


def require_gpu():
    import torch, torchvision
    if not torch.cuda.is_available():
        raise RuntimeError('Enable a GPU accelerator in Kaggle Settings before training.')
    # Also fail early if Kaggle torch/torchvision are incompatible.
    torchvision.ops.nms(torch.tensor([[0.,0.,1.,1.]],device='cuda'),
                        torch.tensor([1.],device='cuda'), .5)
    print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__,
          '| torchvision:', torchvision.__version__)


def obtain_dataset(data_root='', download_root='/kaggle/working/panoptic_v5'):
    if data_root:
        root = Path(data_root)
    else:
        root = Path(download_root)
        if not list(root.rglob('_annotations.coco.json')):
            from kaggle_secrets import UserSecretsClient
            from roboflow import Roboflow
            try:
                key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
            except Exception as exc:
                raise RuntimeError('Add the Kaggle secret ROBOFLOW_API_KEY and enable it for this notebook, '
                                   'or set DATA_ROOT to an attached COCO Segmentation v5 export.') from exc
            # Suppress SDK output: download URLs can contain the API key.
            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    ds = (Roboflow(api_key=key).workspace('panopticindoor')
                          .project('panoptic-indoor-segmentation').version(5)
                          .download('coco-segmentation', location=str(root), overwrite=False))
                root = Path(ds.location)
            except Exception:
                raise RuntimeError('Roboflow download failed. Check Internet and the API-key permission, '
                                   'or attach a manual v5 COCO Segmentation export as DATA_ROOT.') from None
    if not root.exists(): raise FileNotFoundError(root)
    # Accommodate one extra directory level in a Kaggle dataset upload.
    candidates = [p.parent.parent for p in root.rglob('_annotations.coco.json') if p.parent.name=='train']
    if (root/'train/_annotations.coco.json').exists(): return root
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise ValueError(f'Expected one COCO training export under {root}; found {len(candidates)}. Set DATA_ROOT precisely.')
    return candidates[0]


def decode_segmentation(seg, height, width):
    from pycocotools import mask as mu
    if isinstance(seg, list):
        if not seg: raise ValueError('Missing instance polygon; a bounding-box export cannot train segmentation.')
        if any(len(p)<6 or len(p)%2 for p in seg): raise ValueError('Invalid COCO polygon.')
        rle = mu.merge(mu.frPyObjects(seg, height, width))
    elif isinstance(seg, dict):
        rle = mu.frPyObjects(seg, height, width) if isinstance(seg.get('counts'), list) else dict(seg)
        if isinstance(rle.get('counts'), str): rle['counts'] = rle['counts'].encode()
    else:
        raise ValueError('Expected COCO instance polygons/RLE, not a semantic/panoptic PNG export.')
    m = mu.decode(rle)
    if m.ndim == 3: m = np.any(m, axis=2)
    if m.shape != (height, width): raise ValueError('Mask/image dimensions differ.')
    return m.astype(np.uint8)


def encode_mask(mask):
    from pycocotools import mask as mu
    rle = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle['counts'] = rle['counts'].decode('ascii')
    return rle


def category_names(raw):
    used = {a['category_id'] for a in raw['annotations']}
    parents = {c.get('supercategory') for c in raw['categories']}
    # Roboflow often adds an unused parent category, which is not a trainable class.
    cats = [c for c in raw['categories'] if c['id'] in used or c['name'] not in parents]
    names = [c['name'] for c in cats]
    if len(names)!=len(set(names)): raise ValueError('Duplicate class names in COCO categories.')
    return sorted(names)


def find_image(json_path, record):
    base = Path(json_path).parent
    p = base / record['file_name']
    if not p.is_file(): p = base / Path(record['file_name']).name
    if not p.is_file(): raise FileNotFoundError(f'Missing image {record["file_name"]} near {json_path}')
    return p.resolve()


def canonicalize(json_path, names):
    from pycocotools import mask as mu
    raw = read_json(json_path)
    if not raw.get('images'): raise ValueError(f'Empty image split: {json_path}')
    if not raw.get('annotations'): raise ValueError(f'No annotations: {json_path}')
    ids = [x['id'] for x in raw['images']]
    if len(ids)!=len(set(ids)): raise ValueError('Duplicate image IDs.')
    cat = {c['id']:c['name'] for c in raw['categories']}
    name_to_id = {n:i+1 for i,n in enumerate(names)}
    images, paths, hashes = [], {}, {}
    for rec in raw['images']:
        p = find_image(json_path, rec)
        with Image.open(p) as im: w,h = im.size
        if (w,h)!=(rec['width'],rec['height']):
            raise ValueError(f'Image size differs from COCO metadata: {p.name}. Re-export without changing the images.')
        images.append(dict(id=rec['id'],file_name=str(p),width=w,height=h))
        paths[rec['id']] = (w,h)
        hashes[rec['id']] = sha256(p)
    anns=[]
    for ann in raw['annotations']:
        if ann['image_id'] not in paths: raise ValueError('Annotation references an absent image.')
        name=cat[ann['category_id']]
        if name not in name_to_id:
            raise ValueError(f'Unknown class {name!r}. Match the training class spelling exactly; do not silently merge labels.')
        w,h=paths[ann['image_id']]
        m=decode_segmentation(ann.get('segmentation'),h,w)
        rle=mu.encode(np.asfortranarray(m))
        area=float(mu.area(rle)); box=mu.toBbox(rle).tolist()
        if area<=0: raise ValueError(f'Empty instance mask, annotation {ann["id"]}. Repair the source annotation.')
        anns.append(dict(id=len(anns)+1,image_id=ann['image_id'],category_id=name_to_id[name],
                         segmentation=ann['segmentation'],bbox=box,area=area,iscrowd=int(ann.get('iscrowd',0))))
    return dict(info={'description':'Canonical instance masks; IDs remapped by exact class name'},
                licenses=raw.get('licenses',[]),images=images,annotations=anns,
                categories=[dict(id=i+1,name=n) for i,n in enumerate(names)]), hashes


def prepare_data(source_root, prepared_root, strict_v5=True):
    root=Path(source_root); out=Path(prepared_root); out.mkdir(parents=True,exist_ok=True)
    sources={s:root/s/'_annotations.coco.json' for s in ('train','valid','test')}
    if not sources['valid'].exists(): sources['valid']=root/'val/_annotations.coco.json'
    for p in sources.values():
        if not p.exists(): raise FileNotFoundError(f'Missing original split: {p}; do not substitute validation for test.')
    names=category_names(read_json(sources['train']))
    prepared={}; split_hashes={}; counts={}; class_counts={}; duplicate_pairs=[]
    for s,p in sources.items():
        canonical, hashes=canonicalize(p,names)
        counts[s]=len(canonical['images']); split_hashes[s]=set(hashes.values())
        prepared[s]=str(out/f'{s}.json'); write_json(prepared[s],canonical)
        class_counts[s]=dict(Counter(names[a['category_id']-1] for a in canonical['annotations']))
    if strict_v5 and counts != EXPECTED_COUNTS:
        raise ValueError(f'Expected v5 split sizes {EXPECTED_COUNTS}, found {counts}. Check version/export; '
                         'only set STRICT_V5=False for an explicitly documented exception.')
    for a,b in [('train','valid'),('train','test'),('valid','test')]:
        overlap=split_hashes[a]&split_hashes[b]
        if overlap: duplicate_pairs.append({'splits':[a,b],'exact_duplicate_count':len(overlap)})
    # Preserve the assignment splits but expose any source duplication, never resplit silently.
    if duplicate_pairs: print('WARNING: source split duplicates; discuss leakage in the report:',duplicate_pairs)
    fingerprint=hashlib.sha256(json.dumps({'sources':{s:sha256(p) for s,p in sources.items()},
        'image_hashes':{s:sorted(h) for s,h in split_hashes.items()},'classes':names},sort_keys=True).encode()).hexdigest()
    report=dict(dataset_url=DATASET_URL,counts=counts,classes=names,class_counts=class_counts,
                exact_cross_split_duplicates=duplicate_pairs,dataset_fingerprint=fingerprint,
                source_json_hashes={s:sha256(p) for s,p in sources.items()})
    write_json(out/'dataset_report.json',report); write_json(out/'classes.json',names)
    print('Classes:',len(names),'| original splits:',counts)
    return dict(root=str(out),names=names,paths=prepared,fingerprint=fingerprint)


def coco_to_yolo(prepared):
    """Convert polygons only for YOLO training. Evaluate on untouched COCO masks."""
    import yaml
    from ultralytics.data.converter import merge_multi_segment
    root=Path(prepared['root'])/'yolo'; multi=0
    for split,json_path in prepared['paths'].items():
        raw=read_json(json_path); by=defaultdict(list)
        for a in raw['annotations']:
            if not a['iscrowd']: by[a['image_id']].append(a)
        imgdir=root/'images'/split; labdir=root/'labels'/split
        imgdir.mkdir(parents=True,exist_ok=True); labdir.mkdir(parents=True,exist_ok=True)
        for rec in raw['images']:
            source=Path(rec['file_name']); dest=imgdir/f'{rec["id"]}{source.suffix.lower()}'
            if not dest.exists(): dest.symlink_to(source)
            lines=[]
            for a in by[rec['id']]:
                seg=a['segmentation']
                if not isinstance(seg,list):
                    raise ValueError('YOLO conversion needs polygon export. RLE masks cannot be losslessly '
                                     'represented in YOLO polygon labels. Re-export as COCO Segmentation polygons.')
                if len(seg)>1:
                    points=np.concatenate(merge_multi_segment(seg),axis=0); multi+=1
                else: points=np.asarray(seg[0],dtype=float).reshape(-1,2)
                points=points / np.array([rec['width'],rec['height']])
                if not np.isfinite(points).all(): raise ValueError('Non-finite polygon coordinates.')
                if (points < -1e-5).any() or (points > 1.00001).any():
                    raise ValueError('Out-of-image polygon: repair source labels before training.')
                points=np.clip(points,0,1)
                lines.append(str(a['category_id']-1)+' '+' '.join(f'{v:.7f}' for v in points.ravel()))
            (labdir/f'{rec["id"]}.txt').write_text('\n'.join(lines),encoding='utf-8')
    config={'path':str(root.resolve()),'train':'images/train','val':'images/valid','test':'images/test','names':prepared['names']}
    (root/'data.yaml').write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    write_json(root/'conversion_report.json',{'multipolygon_instances_bridged':multi,
        'note':'YOLO uses connected polygons for multipart objects; COCO evaluation retains the original masks. Crowd labels excluded from training.'})
    return str(root/'data.yaml')

### 3.6 Evaluation and previews

In [ ]:
def collect_environment(run_dir):
    import torch, torchvision
    write_json(Path(run_dir)/'environment.json',dict(python=sys.version,torch=torch.__version__,
        torchvision=torchvision.__version__,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        cuda=torch.version.cuda,seed=SEED))
    p=subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True)
    (Path(run_dir)/'pip_freeze.txt').write_text(p.stdout,encoding='utf-8')


def plot_predictions(rec, predictions, categories, out, threshold=.25):
    im=Image.open(rec['file_name']).convert('RGB'); overlay=Image.new('RGBA',im.size,(0,0,0,0)); draw=ImageDraw.Draw(overlay)
    palette=[(255,90,70),(50,200,255),(170,90,255),(255,210,40),(40,225,130)]
    for p in predictions:
        if p['score']<threshold: continue
        m=decode_segmentation(p['segmentation'],rec['height'],rec['width'])
        color=palette[(p['category_id']-1)%len(palette)]
        rgba=Image.new('RGBA',im.size,color+(95,))
        overlay.alpha_composite(Image.composite(rgba,Image.new('RGBA',im.size),Image.fromarray(m*255)))
        x,y,w,h=p['bbox']; draw=ImageDraw.Draw(overlay)
        draw.rectangle([x,y,x+w,y+h],outline=color+(255,),width=2)
        draw.text((x,max(0,y-12)),f'{categories[p["category_id"]]} {p["score"]:.2f}',fill=color+(255,))
    Image.alpha_composite(im.convert('RGBA'),overlay).convert('RGB').save(out)


def evaluate_coco(json_path,predictions,out_dir,visual_limit=12):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    out=Path(out_dir); out.mkdir(parents=True,exist_ok=True)
    gt=COCO(str(json_path)); write_json(out/'predictions_coco.json',predictions)
    # Remove bbox for segmentation loadRes so size-area bins use mask area.
    dt=gt.loadRes([{k:v for k,v in p.items() if k!='bbox'} for p in predictions]) if predictions else COCO()
    if not predictions:
        dt.dataset=dict(images=gt.dataset['images'],categories=gt.dataset['categories'],annotations=[]);dt.createIndex()
    ev=COCOeval(gt,dt,'segm'); ev.params.maxDets=[1,10,100]
    log=io.StringIO()
    with contextlib.redirect_stdout(log): ev.evaluate();ev.accumulate();ev.summarize()
    print(log.getvalue()); (out/'coco_summary.txt').write_text(log.getvalue())
    metrics={k:float(ev.stats[i]) for i,k in enumerate(['mask_AP50_95','mask_AP50','mask_AP75',
        'mask_AP_small','mask_AP_medium','mask_AP_large','mask_AR1','mask_AR10','mask_AR100',
        'mask_AR_small','mask_AR_medium','mask_AR_large'])}
    metrics.update(images=len(gt.imgs),instances=len(gt.anns),max_detections=100)
    write_json(out/'metrics.json',metrics)
    with open(out/'per_class_AP.csv','w',newline='') as f:
        writer=csv.writer(f); writer.writerow(['category_id','class','mask_AP50_95','mask_AP50','gt_instances'])
        for k,cid in enumerate(ev.params.catIds):
            all_ap=ev.eval['precision'][:,:,k,0,-1]; ap50=ev.eval['precision'][0,:,k,0,-1]
            writer.writerow([cid,gt.cats[cid]['name'],float(all_ap[all_ap>=0].mean()) if (all_ap>=0).any() else '',
                             float(ap50[ap50>=0].mean()) if (ap50>=0).any() else '',len(gt.getAnnIds(catIds=[cid]))])
    if visual_limit:
        vis=out/'overlays';vis.mkdir(exist_ok=True); by=defaultdict(list)
        for p in predictions: by[p['image_id']].append(p)
        for rec in list(gt.imgs.values())[:visual_limit]:
            plot_predictions(rec,by[rec['id']],{k:v['name'] for k,v in gt.cats.items()},vis/f'{rec["id"]}.jpg')
    return metrics


def predict_yolo(weights,json_path,imgsz=640):
    from ultralytics import YOLO
    from pycocotools import mask as mu
    model=YOLO(str(weights)); predictions=[]
    for rec in read_json(json_path)['images']:
        r=model.predict(rec['file_name'],imgsz=imgsz,device=0,conf=.001,iou=.7,
                        max_det=100,retina_masks=True,verbose=False)[0]
        if r.masks is None: continue
        for mask,score,cls in zip(r.masks.data.cpu().numpy(),r.boxes.conf.cpu().numpy(),r.boxes.cls.cpu().numpy()):
            if mask.shape!=(rec['height'],rec['width']): raise RuntimeError('YOLO mask was not restored to source resolution.')
            m=(mask>.5).astype(np.uint8); enc=encode_mask(m)
            predictions.append(dict(image_id=rec['id'],category_id=int(cls)+1,score=float(score),
                                    segmentation=enc,bbox=mu.toBbox(enc).tolist()))
    return predictions


def collate(batch): return tuple(zip(*batch))


def preview_maskrcnn_dataset(dataset, classes, output_dir, limit=6):
    """Save transformed image/mask pairs as real augmentation-process evidence."""
    from torchvision.transforms.functional import to_pil_image
    from pycocotools import mask as mu
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True)
    for i in range(min(limit,len(dataset))):
        image,target=dataset[i];path=out/f'{i:02d}_image.jpg';to_pil_image(image).save(path)
        rec=dict(file_name=str(path),height=image.shape[1],width=image.shape[2])
        preds=[]
        for mask,label in zip(target['masks'],target['labels']):
            rle=encode_mask(mask.numpy())
            preds.append(dict(category_id=int(label),score=1.,segmentation=rle,bbox=mu.toBbox(rle).tolist()))
        plot_predictions(rec,preds,{i+1:n for i,n in enumerate(classes)},out/f'{i:02d}_ground_truth_masks.jpg')


def plot_maskrcnn_history(run):
    import matplotlib.pyplot as plt
    with open(Path(run)/'results.csv',newline='') as f: rows=list(csv.DictReader(f))
    if not rows:return
    x=[int(r['epoch']) for r in rows]
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].plot(x,[float(r['train_loss']) for r in rows]);axes[0].set_title('Training loss')
    axes[1].plot(x,[float(r['val_mask_AP50_95']) for r in rows],label='mask AP50–95')
    axes[1].plot(x,[float(r['val_mask_AP50']) for r in rows],label='mask AP50');axes[1].legend()
    axes[1].set_title('Original validation');axes[1].set_ylim(0,1)
    for ax in axes:ax.set_xlabel('Epoch');ax.grid(alpha=.2)
    fig.tight_layout();fig.savefig(Path(run)/'training_curves.png',dpi=150);plt.close(fig)

### 3.7 Model construction and training

In [ ]:
def build_maskrcnn(nclasses,imgsz=640,pretrained=True):
    from torchvision.models.detection import maskrcnn_resnet50_fpn,MaskRCNN_ResNet50_FPN_Weights
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
    # Aspect ratio preserved: shorter side targets imgsz, longer side capped at imgsz.
    model=maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT if pretrained else None,
        weights_backbone=None,min_size=imgsz,max_size=imgsz,box_score_thresh=.001,box_detections_per_img=100)
    model.roi_heads.box_predictor=FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,nclasses+1)
    model.roi_heads.mask_predictor=MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,nclasses+1)
    return model


@torch.inference_mode()
def predict_maskrcnn(model,json_path,device='cuda'):
    from pycocotools import mask as mu
    model.eval();predictions=[]
    ds=CocoInstances(json_path)
    for image,target in ds:
        pred=model([image.to(device)])[0]
        for m,score,label in zip(pred['masks'][:,0].cpu().numpy(),pred['scores'].cpu().numpy(),pred['labels'].cpu().numpy()):
            if score<.001: continue
            enc=encode_mask(m>=.5)
            predictions.append(dict(image_id=int(target['image_id']),category_id=int(label),score=float(score),
                                    segmentation=enc,bbox=mu.toBbox(enc).tolist()))
    return predictions


def save_checkpoint(path,state):
    tmp=Path(str(path)+'.tmp');torch.save(state,tmp);tmp.replace(path)


def train_maskrcnn(prepared,run,regime,epochs=30,imgsz=640,batch=2,workers=2,lr=.0025):
    seed_all();device='cuda';model=build_maskrcnn(len(prepared['names']),imgsz).to(device)
    optimizer=torch.optim.SGD([p for p in model.parameters() if p.requires_grad],lr=lr,momentum=.9,weight_decay=.0005)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=sorted(set([int(epochs*.67),int(epochs*.89)])),gamma=.1)
    scaler=torch.amp.GradScaler('cuda'); generator=torch.Generator().manual_seed(SEED)
    ds=CocoInstances(prepared['paths']['train'],regime,True)
    if not (run/'augmentation_previews').exists():
        preview_maskrcnn_dataset(ds,prepared['names'],run/'augmentation_previews')
    seed_all()  # Preview sampling must not shift the training RNG stream, even after a retry.
    loader=DataLoader(ds,batch_size=batch,shuffle=True,num_workers=workers,collate_fn=collate,
                      pin_memory=True,worker_init_fn=seed_worker,generator=generator,persistent_workers=False)
    start=0; best=-1.; elapsed=0.; last=run/'last.pth'
    if last.exists():
        ck=torch.load(last,map_location='cpu',weights_only=False)  # only your own trusted checkpoint
        model.load_state_dict(ck['model']);optimizer.load_state_dict(ck['optimizer']);scheduler.load_state_dict(ck['scheduler'])
        scaler.load_state_dict(ck['scaler']);start=ck['epoch'];best=ck['best_AP'];elapsed=ck['train_val_seconds']
        random.setstate(ck['rng_python']);np.random.set_state(ck['rng_numpy']);torch.set_rng_state(ck['rng_torch'])
        torch.cuda.set_rng_state_all(ck['rng_cuda']);generator.set_state(ck['rng_loader'])
    logpath=run/'results.csv'
    # Remove any log row written after the latest successfully committed checkpoint.
    if logpath.exists():
        with open(logpath,newline='') as f: rows=list(csv.DictReader(f));fields=list(rows[0]) if rows else []
        if fields:
            with open(logpath,'w',newline='') as f:
                writer=csv.DictWriter(f,fields);writer.writeheader();writer.writerows(r for r in rows if int(r['epoch'])<=start)
    for epoch in range(start,epochs):
        torch.cuda.synchronize();t=time.perf_counter();model.train();totals=defaultdict(float)
        for images,targets in loader:
            images=[x.to(device,non_blocking=True) for x in images]
            targets=[{k:v.to(device) for k,v in target.items()} for target in targets]
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                losses=model(images,targets);loss=sum(losses.values())
            if not torch.isfinite(loss): raise FloatingPointError(f'Nonfinite training loss at epoch {epoch+1}')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),10.);scaler.step(optimizer);scaler.update()
            for k,v in losses.items(): totals[k]+=float(v.detach())
            totals['train_loss']+=float(loss.detach())
        learning_rate=optimizer.param_groups[0]['lr'];scheduler.step()
        metrics=evaluate_coco(prepared['paths']['valid'],predict_maskrcnn(model,prepared['paths']['valid']),run/'validation_latest',0)
        torch.cuda.synchronize();seconds=time.perf_counter()-t;elapsed+=seconds
        row=dict(epoch=epoch+1,**{k:v/len(loader) for k,v in totals.items()},
                 val_mask_AP50_95=metrics['mask_AP50_95'],val_mask_AP50=metrics['mask_AP50'],lr=learning_rate,
                 epoch_seconds=seconds,train_val_seconds=elapsed)
        with open(logpath,'a',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(row))
            if f.tell()==0: writer.writeheader()
            writer.writerow(row)
        score=metrics['mask_AP50_95']
        if score>best:
            best=score
            save_checkpoint(run/'best.pth',dict(model=model.state_dict(),classes=prepared['names'],
                                               epoch=epoch+1,imgsz=imgsz,best_AP=best))
        save_checkpoint(last,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),
            scaler=scaler.state_dict(),epoch=epoch+1,best_AP=best,train_val_seconds=elapsed,classes=prepared['names'],imgsz=imgsz,
            rng_python=random.getstate(),rng_numpy=np.random.get_state(),rng_torch=torch.get_rng_state(),
            rng_cuda=torch.cuda.get_rng_state_all(),rng_loader=generator.get_state()))
        print(f'Epoch {epoch+1}/{epochs}: loss={row["train_loss"]:.4f}, val mask AP={score:.4f}, {seconds:.1f}s')
    del model;torch.cuda.empty_cache()
    plot_maskrcnn_history(run)
    return run/'best.pth',elapsed


def completed_yolo_fit_time(run, epochs):
    """Recover a completed fit if interruption occurred before wrapper bookkeeping."""
    run=Path(run);last=run/'fit/weights/last.pt';best=run/'fit/weights/best.pt';csvpath=run/'fit/results.csv'
    if not all(p.is_file() for p in (last,best,csvpath)): return None
    with open(csvpath,newline='') as f:
        rows=[{k.strip():v for k,v in r.items()} for r in csv.DictReader(f)]
    if not rows or int(float(rows[-1]['epoch']))<epochs: return None
    ck=torch.load(last,map_location='cpu',weights_only=False)  # own trusted checkpoint only
    epoch=ck.get('epoch',-2)
    committed=epoch>=epochs-1 or (epoch==-1 and ck.get('optimizer') is None)
    return float(rows[-1]['time']) if committed else None


def train_yolo(prepared,run,model_name,regime,epochs=30,imgsz=640,batch=4,workers=2,lr=.001):
    import ultralytics
    from ultralytics import YOLO
    if ultralytics.__version__!='8.3.221': raise RuntimeError('Install the pinned ultralytics==8.3.221 version.')
    data=coco_to_yolo(prepared);seed_all();last=run/'fit/weights/last.pt'
    # A completed fit can be evaluated again without asking YOLO to resume a stripped checkpoint.
    fitdone=run/'fit_complete.json'
    if fitdone.exists(): return run/'fit/weights/best.pt',read_json(fitdone)['train_val_seconds']
    recovered_time=completed_yolo_fit_time(run,epochs)
    if recovered_time is not None:
        write_json(fitdone,dict(train_val_seconds=recovered_time,recovered_completed_fit=True))
        return run/'fit/weights/best.pt',recovered_time
    kwargs=dict(data=data,epochs=epochs,imgsz=imgsz,batch=batch,device=0,workers=workers,
                optimizer='AdamW',lr0=lr,lrf=.01,weight_decay=.0005,seed=SEED,deterministic=True,
                patience=0,amp=True,cache=False,save=True,save_period=5,plots=True,
                project=str(run),name='fit',exist_ok=True,pretrained=True,
                multi_scale=False,rect=False,**YOLO_AUG[regime])
    # Keep augmentation policy active during dataset construction, including a resumed run.
    with yolo_augmentation_policy(regime):
        model=YOLO(str(last) if last.exists() else model_name+'.pt')
        torch.cuda.synchronize();t=time.perf_counter()
        if last.exists(): model.train(resume=True,device=0,workers=workers)
        else: model.train(**kwargs)
        torch.cuda.synchronize();session_seconds=time.perf_counter()-t
    # Native CSV time is cumulative within YOLO's recorded fit (including epoch validation).
    with open(run/'fit/results.csv',newline='') as f: rows=[{k.strip():v for k,v in r.items()} for r in csv.DictReader(f)]
    native_seconds=float(rows[-1].get('time',session_seconds))
    write_json(fitdone,dict(train_val_seconds=native_seconds,current_session_seconds=session_seconds,
                          note='Native results.csv time; interrupted YOLO sessions may reset its time column. See session ledger.'))
    del model;torch.cuda.empty_cache()
    return run/'fit/weights/best.pt',native_seconds

### 3.8 Experiment runner and output helpers

In [ ]:
def run_experiment(prepared,out_root,model_name,regime,epochs=30,imgsz=640,
                   yolo_batch=4,mask_batch=2,workers=2):
    require_gpu()
    if model_name not in MODELS or regime not in VALID_REGIMES: raise ValueError('Unknown model/augmentation policy.')
    if not 1<=epochs<=1000: raise ValueError('Invalid epoch count.')
    run=Path(out_root)/f'{model_name}_{regime}';run.mkdir(parents=True,exist_ok=True)
    config=dict(model=model_name,regime=regime,epochs=epochs,imgsz=imgsz,seed=SEED,
                batch=mask_batch if model_name=='maskrcnn' else yolo_batch,
                dataset_fingerprint=prepared['fingerprint'],classes=prepared['names'],
                augmentation=YOLO_AUG[regime] if model_name!='maskrcnn' or regime=='custom' else regime,
                selection='Original validation set only; no test-based checkpoint selection')
    if (run/'config.json').exists() and read_json(run/'config.json')!=config:
        raise ValueError(f'Run settings changed: {run}. Use a new OUT_ROOT instead of mixing incompatible experiments.')
    write_json(run/'config.json',config)
    for filename in ('dataset_report.json','classes.json'):
        shutil.copy2(Path(prepared['root'])/filename,run/filename)
    if (run/'completed.json').exists():
        print('Already complete:',run.name);return read_json(run/'completed.json')
    collect_environment(run);session_start=time.time();status='interrupted_or_failed'
    try:
        if model_name=='maskrcnn':
            weights,seconds=train_maskrcnn(prepared,run,regime,epochs,imgsz,mask_batch,workers)
            ck=torch.load(weights,map_location='cpu',weights_only=False)
            model=build_maskrcnn(len(prepared['names']),imgsz,False).cuda();model.load_state_dict(ck['model'])
            t=time.perf_counter();pred=predict_maskrcnn(model,prepared['paths']['test'])
            del model
        else:
            weights,seconds=train_yolo(prepared,run,model_name,regime,epochs,imgsz,yolo_batch,workers)
            t=time.perf_counter();pred=predict_yolo(weights,prepared['paths']['test'],imgsz)
        metrics=evaluate_coco(prepared['paths']['test'],pred,run/'original_test')
        result=dict(model=model_name,regime=regime,epochs=epochs,weights=str(weights),
                    train_val_seconds=seconds,test_seconds=time.perf_counter()-t,**metrics)
        write_json(run/'completed.json',result);status='complete'
        return result
    finally:
        torch.cuda.empty_cache()
        with open(run/'session_ledger.jsonl','a') as f:
            f.write(json.dumps(dict(start_unix=session_start,end_unix=time.time(),
                allocated_session_seconds=time.time()-session_start,gpu_count=1,status=status))+'\n')


def prepare_own_test(json_path,run_dir,out_dir):
    names=read_json(Path(run_dir)/'classes.json');canonical,hashes=canonicalize(json_path,names)
    if len(canonical['images'])<40: print('WARNING: assignment requires at least 40 own photos; this export has',len(canonical['images']))
    out=Path(out_dir);out.mkdir(parents=True,exist_ok=True);p=out/'own_test_canonical.json';write_json(p,canonical)
    write_json(out/'own_test_manifest.json',dict(source=str(json_path),source_sha256=sha256(json_path),image_hashes=hashes,
                                               classes=names,note='Never used in training or checkpoint selection.'))
    return p


def evaluate_own(run_dir,own_json,out_dir=None,imgsz_override=None):
    require_gpu();run=Path(run_dir);cfg=read_json(run/'config.json')
    out=Path(out_dir) if out_dir else Path('/kaggle/working/own_evaluation')/run.name
    json_path=prepare_own_test(own_json,run,out);size=imgsz_override or cfg['imgsz']
    if cfg['model']=='maskrcnn':
        ck=torch.load(run/'best.pth',map_location='cpu',weights_only=False)
        if ck['classes']!=cfg['classes']: raise ValueError('Checkpoint class mapping mismatch.')
        model=build_maskrcnn(len(cfg['classes']),size,False).cuda();model.load_state_dict(ck['model'])
        pred=predict_maskrcnn(model,json_path);del model
    else: pred=predict_yolo(run/'fit/weights/best.pt',json_path,size)
    metrics=evaluate_coco(json_path,pred,out,visual_limit=100000)
    if (run/'original_test/metrics.json').exists() and not imgsz_override:
        original=read_json(run/'original_test/metrics.json')
        write_json(out/'domain_shift.json',dict(original_AP=original['mask_AP50_95'],own_AP=metrics['mask_AP50_95'],
            drop_percentage_points=100*(original['mask_AP50_95']-metrics['mask_AP50_95'])))
    write_json(out/'evaluation_config.json',dict(run=str(run),imgsz=size,diagnostic_override=bool(imgsz_override),
        warning='A changed input size on own test is a post-hoc diagnostic, not the primary held-out score.' if imgsz_override else ''))
    torch.cuda.empty_cache();return metrics


def comparison_csv(out_root):
    rows=[]
    for p in sorted(Path(out_root).glob('*/completed.json')):
        d=read_json(p);rows.append({k:d[k] for k in ('model','regime','epochs','mask_AP50_95','mask_AP50','mask_AP75','train_val_seconds','test_seconds')})
    if not rows: return None
    dest=Path(out_root)/'comparison_original_test.csv'
    with open(dest,'w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    return dest


def archive_run(run_dir,destination):
    run=Path(run_dir);destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)
    return shutil.make_archive(str(destination.with_suffix('')),'zip',root_dir=run.parent,base_dir=run.name)

In [ ]:
REFERENCE_CLASSES = ['backpack', 'bed', 'blanket', 'book', 'bottle', 'bowl', 'box', 'cabinet-merged', 'cardboard', 'ceiling-merged', 'cell phone', 'chair', 'clock', 'couch', 'counter', 'cup', 'curtain', 'dining table', 'door-stuff', 'fire-extinguisher', 'floor-other-merged', 'floor-wood', 'flower', 'food-other-merged', 'fork', 'hair drier', 'handbag', 'keyboard', 'knife', 'laptop', 'light', 'microwave', 'mirror-stuff', 'mouse', 'oven', 'pan', 'paper', 'paper-merged', 'person', 'pillow', 'plate', 'plug', 'potted plant', 'refrigerator', 'remote', 'rug-merged', 'scissor', 'scissors', 'shelf', 'sink', 'spoon', 'st\\', 'stairs', 'stove-merged', 'table-merged', 'teddy bear', 'toaster', 'toilet', 'toothbrush', 'towel', 'toy', 'trash-bin', 'tree-merged', 'tv', 'vase', 'wall', 'wall-brick', 'wall-other-merged', 'wall-stone', 'wall-tile', 'wall-wood', 'wallet', 'window-blind', 'window-other']
print("Notebook helpers and all three augmentation policies are ready.")

In [ ]:
# Skip images whose actual dimensions disagree with their annotations.
# Valid landscape/portrait images are both retained.

# Allow split counts to decrease after skipping mismatched images.
STRICT_V5 = False

# Preserve the original function, including when this cell is rerun.
if "_canonicalize_before_skip" not in globals():
    _canonicalize_before_skip = canonicalize


def canonicalize(json_path, names):
    import hashlib
    from pathlib import Path
    from PIL import Image

    raw = read_json(json_path)
    kept, skipped, skipped_ids = [], [], set()

    for record in raw["images"]:
        path = find_image(json_path, record)

        with Image.open(path) as image:
            actual = image.size

        expected = (record["width"], record["height"])

        if actual != expected:
            skipped_ids.add(record["id"])
            skipped.append({
                "file": path.name,
                "actual": actual,
                "expected": expected,
            })
            continue

        kept.append(dict(record, file_name=str(path)))

    # Save a record of excluded images without modifying the originals.
    folder = Path("/kaggle/working/seg046_size_checks")
    folder.mkdir(parents=True, exist_ok=True)

    tag = hashlib.sha256(
        str(Path(json_path).resolve()).encode()
    ).hexdigest()[:12]

    write_json(folder / f"{tag}_skipped.json", skipped)

    print(
        f"{Path(json_path).parent.name}: "
        f"kept {len(kept)}, skipped {len(skipped)}"
    )

    if not kept:
        raise ValueError(
            "No valid images remain in this split. Check the dataset export."
        )

    raw["images"] = kept
    raw["annotations"] = [
        annotation
        for annotation in raw["annotations"]
        if annotation["image_id"] not in skipped_ids
    ]

    filtered = folder / f"{tag}_filtered.coco.json"
    write_json(filtered, raw)

    return _canonicalize_before_skip(filtered, names)

## 4. Preflight and dataset validation

In [ ]:
require_gpu()
source_root = obtain_dataset(DATA_ROOT)
prepared = prepare_data(source_root, PREPARED_ROOT, strict_v5=STRICT_V5)
# Report tag-list discrepancies without silently changing the actual export's labels.
reference = set(REFERENCE_CLASSES)
actual = set(prepared["names"])
print("Reference tags absent from export:", sorted(reference - actual))
print("Export classes absent from reference tags:", sorted(actual - reference))
print("Resolved exact class order:", prepared["names"])
from IPython.display import display
import pandas as pd
report = read_json(Path(PREPARED_ROOT) / "dataset_report.json")
display(pd.DataFrame(report["class_counts"]).fillna(0).astype(int))


## 5. Optional restore from a previous session

In [ ]:
import shutil
Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
if RESTORE_RUNS_FROM:
    old_root = Path(RESTORE_RUNS_FROM)
    allowed = {f"{model}_{regime}" for model in MODELS for regime in VALID_REGIMES}
    candidates = [p.parent for p in old_root.rglob("config.json") if p.parent.name in allowed]
    if not candidates: raise FileNotFoundError("No saved run folders found under RESTORE_RUNS_FROM.")
    for folder in candidates:
        dest = Path(OUT_ROOT) / folder.name
        if dest.exists():
            print("Keeping existing writable run:", dest)
        else:
            shutil.copytree(folder, dest)
            print("Restored:", dest)


## 6. Train, evaluate original test, save each result

In [ ]:
from IPython.display import display, FileLink
import time
zip_dir = Path("/kaggle/working/result_zips")
zip_dir.mkdir(exist_ok=True)
results = []
for regime in REGIMES:
    print(f"\n===== {MODEL} | {regime} | {EPOCHS} epochs =====")
    run_dir = Path(OUT_ROOT) / f"{MODEL}_{regime}"
    try:
        result = run_experiment(prepared, OUT_ROOT, MODEL, regime,
            epochs=EPOCHS, imgsz=IMGSZ, yolo_batch=YOLO_BATCH,
            mask_batch=MASK_BATCH, workers=WORKERS)
        results.append(result)
    finally:
        # Make a partial checkpoint archive even if this Python run raises an error.
        # A forced process/session kill can bypass this block; retain Kaggle outputs.
        if run_dir.exists():
            archive = archive_run(run_dir, zip_dir / f"{run_dir.name}.zip")
            print("Saved", archive)
            display(FileLink(archive))
    table = comparison_csv(OUT_ROOT)
    if table:
        display(pd.read_csv(table))
print("Requested runs finished. Save this notebook version and download the result ZIPs.")


## Keep these outputs

Download every ZIP in `/kaggle/working/result_zips`, or retain the full saved notebook outputs. Each contains checkpoints, training logs, configuration, class mapping, provenance and original-test scores. `completed.json` means the complete fit + original-test evaluation finished; a partial ZIP is not a completed experiment.

Your own photos are intentionally absent. When their polygons are ready, use **04_Evaluate_Own_Photos_Later.ipynb** with these saved run folders. See README for the ≥40 photos, ≥10 difficult, annotation screenshots, EXIF, five failure cases and AI declaration requirements.

The scores above are measured output only after this notebook actually runs. The delivered notebook contains no invented accuracy.
